# ⚽ FIFA World Cup Intelligence & Prediction System
## Phase 1: Data Understanding & Exploratory Analysis


### 1: Import Libraries

In [49]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Settings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

sns.set_theme(style='whitegrid')

print("Libraries loaded successfully.")

Libraries loaded successfully.


### 2: Load Data

In [9]:
# Load the main results file
df = pd.read_csv('../data/results.csv')

# Parse date column
df['date'] = pd.to_datetime(df['date'])

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')


Dataset loaded: 49477 rows, 9 columns
First 5 rows:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.00,0.00,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.00,2.00,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.00,1.00,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.00,2.00,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.00,0.00,Friendly,Glasgow,Scotland,False


In [10]:
# Random sample
print('Random sample (10 rows):')
df.sample(10, random_state=42)

Random sample (8 rows):


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
45400,2022-06-13,Guatemala,Dominican Republic,2.00,0.00,CONCACAF Nations League,Guatemala City,Guatemala,False
2791,1945-10-21,Sweden,Norway,10.00,0.00,Friendly,Solna,Sweden,False
46584,2023-10-16,Botswana,Eswatini,2.00,1.00,Friendly,Lobatse,Botswana,False
199,1907-02-23,Northern Ireland,Wales,2.00,3.00,British Home Championship,Belfast,Ireland,False
45518,2022-09-23,Saudi Arabia,Ecuador,0.00,0.00,Friendly,Murcia,Spain,True
6556,1966-02-27,Niger,DR Congo,2.00,1.00,Friendly,Niamey,Niger,False
7100,1967-08-13,India,Western Australia,3.00,1.00,Merdeka Tournament,Kuala Lumpur,Malaysia,True
36297,2012-10-12,Czech Republic,Malta,3.00,1.00,FIFA World Cup qualification,Plzeň,Czech Republic,False



### 4: Dataset Overview

In [12]:
# Professional summary table
summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Unique Values': df.nunique().values
})

summary['Null %'] = (summary['Null Count'] / len(df) * 100).round(2)
summary = summary.reset_index(drop=True)
summary

,Column,Data Type,Non-Null Count,Null Count,Unique Values,Null %
0,date,datetime64[ns],49477,0,16471,0.00
1,home_team,object,49477,0,327,0.00
2,away_team,object,49477,0,321,0.00
3,home_score,float64,49425,52,26,0.11
4,away_score,float64,49425,52,22,0.11
5,tournament,object,49477,0,200,0.00
6,city,object,49477,0,2089,0.00
7,country,object,49477,0,269,0.00
8,neutral,bool,49477,0,2,0.00



### 5: Data Quality Assessment

In [17]:
# --- Duplicate Rows ---
duplicates = df.duplicated().sum()
print(f'Duplicate rows : {duplicates}')

Duplicate rows : 0


In [ ]:
# Check score ranges for any obvious anomalies
print('Score range summary:')
df[['home_score', 'away_score']].describe()


### 6: Univariate Analysis

In [20]:
# --- Q1: Total matches ---
total_matches = len(df)
print(f'Total matches in dataset: {total_matches}')

Total matches in dataset: 49477


In [21]:
# --- Q2: Dataset time range ---
print(f'Earliest match : {df["date"].min().date()}')
print(f'Latest match   : {df["date"].max().date()}')
print(f'Span           : {df["date"].max().year - df["date"].min().year} years')

Earliest match : 1872-11-30
Latest match   : 2026-06-27
Span           : 154 years


In [25]:
# --- Q3: Unique countries ---
all_teams = pd.concat([df['home_team'], df['away_team']])
unique_countries = all_teams.nunique()
print(f'Unique countries/teams: {unique_countries}')

Unique countries/teams: 336


In [ ]:
# --- Q4 & Q5: Tournaments ---
unique_tournaments = df['tournament'].nunique()
print(f'Unique tournaments: {unique_tournaments}')

print('\nTop 15 most frequent tournaments:')
tournament_counts = df['tournament'].value_counts().head(15)
print(tournament_counts)

# Plot 
top10_tournaments = df['tournament'].value_counts().head(10)

sns.barplot(
    x=top10_tournaments.values,
    y=top10_tournaments.index
)

plt.title('Top 10 Most Frequent Tournaments')
plt.xlabel('Number of Matches')
plt.show()

In [105]:
# --- Q6: Most frequently appearing countries ---
team_counts = all_teams.value_counts().head(15).reset_index()
print(team_counts.columns)
fig = px.bar(
    team_counts,
    x='count',
    y='index',
    text='count',
    title='Top 15 Countries by Total Appearances'
)
fig.show()

Index(['index', 'count'], dtype='object')



### 7: Temporal Analysis

In [ ]:
# Matches per year
df['year'] = df['date'].dt.year
matches_per_year = df.groupby('year').size().reset_index(name="matches")
print(matches_per_year.tail())

# Plot 
fig = px.line(
    matches_per_year,
    x='year',
    y='matches',
    title='Matches Played Per Year'
)
fig.show()


In [ ]:
# Matches per decade
df['decade'] = (df['year'] // 10) * 10
matches_per_decade = df.groupby('decade').size().reset_index(name='matches')

# Plot 
fig = px.line(
    matches_per_decade,
    x='decade',
    y='matches',
    title='Matches Played Per Decade'
)
fig.show()



In [65]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,decade
0,1872-11-30,Scotland,England,0.00,0.00,Friendly,Glasgow,Scotland,False,1872,1870
1,1873-03-08,England,Scotland,4.00,2.00,Friendly,London,England,False,1873,1870
2,1874-03-07,Scotland,England,2.00,1.00,Friendly,Glasgow,Scotland,False,1874,1870
3,1875-03-06,England,Scotland,2.00,2.00,Friendly,London,England,False,1875,1870
4,1876-03-04,Scotland,England,3.00,0.00,Friendly,Glasgow,Scotland,False,1876,1870



### 8: Goal Scoring Analysis

In [70]:
# Drop rows with missing scores for this section
df_scores = df.dropna(subset=['home_score', 'away_score']).copy()
df_scores['total_goals'] = df_scores['home_score'] + df_scores['away_score']

print('Goal scoring summary:')
print(df_scores[['home_score', 'away_score', 'total_goals']].describe())

Goal scoring summary:
       home_score  away_score  total_goals
count    49425.00    49425.00     49425.00
mean         1.76        1.18         2.94
std          1.77        1.40         2.10
min          0.00        0.00         0.00
25%          1.00        0.00         1.00
50%          1.00        1.00         3.00
75%          2.00        2.00         4.00
max         31.00       21.00        31.00


In [71]:
# Average goals
avg_home = df_scores['home_score'].mean()
avg_away = df_scores['away_score'].mean()
avg_total = df_scores['total_goals'].mean()

print(f'Avg home goals  : {avg_home:.2f}')
print(f'Avg away goals  : {avg_away:.2f}')
print(f'Avg total goals : {avg_total:.2f}')

Avg home goals  : 1.76
Avg away goals  : 1.18
Avg total goals : 2.94


In [ ]:
# Histograms — home, away, total goals 
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(15, 5))

ax0.hist(df_scores['home_score'], bins=range(0, 20), color='steelblue')
ax0.set_title('Home Goals')
ax0.set_xlabel('Goals')
ax0.set_ylabel('Matches')

ax1.hist(df_scores['away_score'], bins=range(0, 20), color='coral')
ax1.set_title('Away Goals')
ax1.set_xlabel('Goals')
ax1.set_ylabel('Matches')

ax2.hist(df_scores['total_goals'], bins=range(0, 20), color='mediumseagreen')
ax2.set_title('Total Goals')
ax2.set_xlabel('Goals')
ax2.set_ylabel('Matches')

plt.suptitle('Goal Scoring Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot — home vs away goals 

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([
    df_scores['home_score'], df_scores['away_score']
    ])
ax.set_title('Home vs Away Goals — Boxplot',fontweight='bold')
ax.set_xticklabels(['Home Goals', 'Away Goals'])
ax.set_ylabel('Goals')
plt.tight_layout()
plt.show()

In [85]:
# Highest scoring matches
print('Top 10 highest scoring matches:')
df_scores.nlargest(10, 'total_goals')[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'total_goals', 'tournament']]

Top 10 highest scoring matches:


,date,home_team,away_team,home_score,away_score,total_goals,tournament
25425,2001-04-11,Australia,American Samoa,31.00,0.00,31.00,FIFA World Cup qualification
8551,1971-09-13,Tahiti,Cook Islands,30.00,0.00,30.00,South Pacific Games
11916,1979-08-30,Fiji,Kiribati,24.00,0.00,24.00,South Pacific Games
25422,2001-04-09,Australia,Tonga,22.00,0.00,22.00,FIFA World Cup qualification
30518,2006-11-24,Sápmi,Monaco,21.00,1.00,22.00,Viva World Cup
37061,2013-06-24,Provence,Tibet,22.00,0.00,22.00,"International Tournament of Peoples, Cultures ..."
6580,1966-04-03,Libya,Oman,21.00,0.00,21.00,Arab Cup
21957,1997-05-13,Kazakhstan,Guam,20.00,1.00,21.00,East Asian Games
29045,2005-03-11,Guam,North Korea,0.00,21.00,21.00,EAFF Championship
37059,2013-06-23,Quebec,Tibet,21.00,0.00,21.00,"International Tournament of Peoples, Cultures ..."



### 9: Tournament Analysis

In [100]:
# Top 20 tournaments by match count
top20_tournaments = df['tournament'].value_counts().head(20)
print('Top 20 tournaments by number of matches:')
print(top20_tournaments)

Top 20 tournaments by number of matches:
                              tournament  count
0                               Friendly  18388
1           FIFA World Cup qualification   8771
2                UEFA Euro qualification   2824
3   African Cup of Nations qualification   2327
4                         FIFA World Cup   1036
5                           Copa América    869
6                 African Cup of Nations    845
7            AFC Asian Cup qualification    829
8                    UEFA Nations League    658
9                             CECAFA Cup    620
10       CFU Caribbean Cup qualification    606
11                    Merdeka Tournament    599
12             British Home Championship    523
13               CONCACAF Nations League    422
14                         AFC Asian Cup    421
15                              Gold Cup    420
16                              Gulf Cup    410
17                          Island Games    394
18                             UEFA Euro    388

In [ ]:
# Top 10 bar chart
top10 = df['tournament'].value_counts().head(10).reset_index()
print(top10.columns)
fig = px.bar(
    top10,
    x='count',
    y='tournament',
    text='count',
    title='Top 10 Tournaments by Match Count'
)
fig.show()



### 10: Country Participation Analysis

In [ ]:
# Home, away, and total appearances per country
home_appearances = df['home_team'].value_counts()
away_appearances = df['away_team'].value_counts()

country_participation = pd.DataFrame(['home_appearances','away_appearances']).fillna(0)
country_participation = pd.DataFrame({
    'home_appearances': home_appearances,
    'away_appearances': away_appearances
}).fillna(0).astype(int)
country_participation['total_appearances'] = country_participation['home_appearances'] + country_participation['away_appearances']
country_participation = country_participation.sort_values('total_appearances', ascending=False)

print('Top 20 most active countries:')
country_participation.head(20)

ValueError: invalid literal for int() with base 10: 'home_appearances'